In [44]:
# Imports section
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
import numpy as np
from sklearn.preprocessing import PolynomialFeatures

## Part 1. Loading the dataset

In [45]:
# Using pandas load the dataset (load remotely, not locally)
# Output the first 15 rows of the data
# Display a summary of the table information (number of datapoints, etc.)

science_df = pd.read_csv('science_data_large.csv')
print(science_df.head(15))
print(science_df.describe())

    Temperature °C  Mols KCL     Size nm^3
0              469       647  6.244743e+05
1              403       694  5.779610e+05
2              302       975  6.196847e+05
3              779       916  1.460449e+06
4              901        18  4.325726e+04
5              545       637  7.124634e+05
6              660       519  7.006960e+05
7              143       869  2.718260e+05
8               89       461  8.919803e+04
9              294       776  4.770210e+05
10             991       117  2.441771e+05
11             307       781  5.006455e+05
12             206        70  3.145200e+04
13             437       599  5.390215e+05
14             566        75  9.185271e+04
       Temperature °C     Mols KCL     Size nm^3
count     1000.000000  1000.000000  1.000000e+03
mean       500.500000   471.530000  5.086111e+05
std        288.819436   288.482872  4.474838e+05
min          1.000000     1.000000  1.611429e+01
25%        250.750000   226.750000  1.298267e+05
50%        500.500

## Part 2. Splitting the dataset

In [46]:
# Take the pandas dataset and split it into our features (X) and label (y)
X = science_df.drop(columns=['Size nm^3'])
y = science_df['Size nm^3']

# Use sklearn to split the features and labels into a training/test set. (90% train, 10% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

## Part 3. Perform a Linear Regression

In [47]:
# Use sklearn to train a model on the training set
reg = LinearRegression().fit(X_train, y_train)

# Create a sample datapoint and predict the output of that sample with the trained model
sample_x = np.array([[500, 500]])
sample_y_pred = reg.predict(sample_x)
print(f"predicted output: {sample_y_pred}")

# Report on the score for that model, in your own words (markdown, not code) explain what the score means
print(f"score on test dataset: {reg.score(X_test, y_test)}")

# Extract the coefficents and intercept from the model and write an equation for your h(x) using LaTeX
coefficients = reg.coef_
intercept = reg.intercept_
print(f"coefficients: {coefficients}, intercept: {intercept}")

predicted output: [540029.26034545]
score on test dataset: 0.8552472077276095
coefficients: [ 866.14641337 1032.69506649], intercept: -409391.47958340775


/opt/anaconda3/envs/cs589/lib/python3.10/site-packages/sklearn/base.py:464: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


### Linear Regression model for slime size
Let $x_1$ be the temperature in celsius, $x_2$ the moles of KCL, and $h(x_1, x_2)$ the predicted slime size (in $\text{nm}^3$). Our linear regression learned the following equation for slime size:
$$h(x_1, x_2) = 866.14641337 x_1 + 1032.69506649 x_2 - 409391.47958340775$$

### $R^2$ score of the model
The $R^2$ score of the linear regression model above on the test dataset is $0.8552472077276095$. This score approaches $1$ as the sum of the squared errors between true and predicted slime sizes in the test dataset approaches zero, and approaches $-\infty$ as the sum of the squared errors between true and predicted slime sizes in the test dataset approaches $+\infty$. The score of our slime model is close to $1$, which means that the sum of the squared errors between true and predicted slime sizes in the test dataset is reasonably low.

## Part 4. Use Cross Validation

In [67]:
# Use the cross_val_score function to repeat your experiment across many shuffles of the data
reg = LinearRegression()
cv_scores = cross_val_score(reg, X, y, cv=10)
print(cv_scores)
print(np.mean(cv_scores), np.std(cv_scores))
# Report on their finding and their significance

[0.81123596 0.86440978 0.87808742 0.86561069 0.87495621 0.84484397
 0.87941022 0.86349411 0.78353682 0.88686516]
0.8552450341984701 0.03152876296534241


### $R^2$ scores after cross validation 
After running 10-fold cross-validation, e.g. 10 different shuffles of the data with 90% training & 10% test data, the linear regression model for slime had an average $R^2$ score of $0.8552450341984701$, and a standard deviation of $0.03152876296534241$. The $R^2$ scores were quite close to $1$ on average. From these results, I conclude that linear regression is an effective model for slime size in terms of temperature and moles of KCL.

## Part 5. Using Polynomial Regression

In [76]:
# Using the PolynomialFeatures library perform another regression on an augmented dataset of degree 2
poly = PolynomialFeatures(2)
X_train_poly = poly.fit_transform(X_train)
reg = LinearRegression().fit(X_train_poly, y_train)

# Report on the metrics and output the resultant equation as you did in Part 3.
X_test_poly = poly.fit_transform(X_test)
print(f"polynomial regression score: {reg.score(X_test_poly, y_test)}")
coefficients = reg.coef_
intercept = reg.intercept_
print(f"coefficients: {coefficients}, intercept: {intercept}")

polynomial regression score: 1.0
coefficients: [ 0.00000000e+00  1.20000000e+01 -1.27195512e-07  1.26494371e-11
  2.00000000e+00  2.85714287e-02], intercept: 2.047792077064514e-05


### Polynomial Regression results
Let $x_1$ be the temperature in celsius, $x_2$ the moles of KCL, and $h(x_1, x_2)$ the predicted slime size (in $\text{nm}^3$). The polynomial regression learned the following equation for slime size:
$$h(x_1, x_2) = 12 x_1 + -1.2720 * 10^{-7} x_2  + 1.2649 * 10^{-11} x_1^2 + 2 x_1 x_2 + 2.8571 * 10^{-2} * x_2^2 + 2.0478 * 10^{-5}$$

### Polynomial Regression $R^2$ score
The $R^2$ score of the polynomial regression was exactly $1$, meaning that the slime equation is governed exactly by the equation learned by the polynomial regression (e.g. the error between the test sizes and predicted slime sizes for test datapoints was zero). This suggests that both the training and test datasets were created by sampling from this exact polynomial. A way to check that this model holds in 'reality' would be to run the experiment multiple times with a real slime, and measure the slime sizes after the experiment, and calculate the $R^2$ score of the model on those test datapoints.